# 🔤 Day 07: Trie (Prefix Tree) — The String Search Structure

---

## What's a Trie?

A trie (pronounced "try") stores strings character by character. Each node represents a character, and paths from root to marked nodes form complete words.

```
Words: ["app", "apple", "ape", "bat"]

      (root)
      /    \
     a      b
     |      |
     p      a
    / \     |
   p*  e*   t*
   |
   l
   |
   e*         (* = end of word)
```

**Why use a trie instead of a set/hashmap?**
- Prefix matching: "find all words starting with 'ap'" → O(prefix length)
- Autocomplete / type-ahead search
- Spell checking
- Word search in grids

---

## 🎯 Problem 1: Implement Trie (LC #208)

Build a trie supporting `insert`, `search`, and `startsWith`.

Each node has a dictionary of children (char → node) and an `is_end` flag.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}  # char → TrieNode
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()
    
    def insert(self, word):
        """Add word to trie. O(len(word))"""
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True
    
    def search(self, word):
        """Is exact word in trie? O(len(word))"""
        node = self._find_prefix(word)
        return node is not None and node.is_end
    
    def startsWith(self, prefix):
        """Is any word starting with prefix? O(len(prefix))"""
        return self._find_prefix(prefix) is not None
    
    def _find_prefix(self, prefix):
        """Navigate to the node at end of prefix, or None."""
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return None
            node = node.children[ch]
        return node

# Test it
trie = Trie()
trie.insert("apple")
print(trie.search("apple"))     # True
print(trie.search("app"))       # False (not marked as end)
print(trie.startsWith("app"))   # True
trie.insert("app")
print(trie.search("app"))       # True (now it's a word)

---

## 🎯 Problem 2: Design Add and Search Words (LC #211)

Like a trie but `search` supports `.` as wildcard (matches any character).

**Approach:** Normal trie traversal, but when you hit `.`, try ALL children (backtracking/DFS).

In [ ]:
class WordDictionary:
    def __init__(self):
        self.root = TrieNode()
    
    def addWord(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True
    
    def search(self, word):
        """Search with '.' wildcard support."""
        def dfs(node, i):
            if i == len(word):
                return node.is_end
            
            ch = word[i]
            if ch == '.':
                # Try every child
                for child in node.children.values():
                    if dfs(child, i + 1):
                        return True
                return False
            else:
                if ch not in node.children:
                    return False
                return dfs(node.children[ch], i + 1)
        
        return dfs(self.root, 0)

wd = WordDictionary()
wd.addWord("bad")
wd.addWord("dad")
wd.addWord("mad")
print(wd.search(".ad"))    # True (bad, dad, mad all match)
print(wd.search("b.."))    # True (bad matches)
print(wd.search("b.d"))    # True
print(wd.search("..."))    # True
print(wd.search("...."))   # False (no 4-letter words)

---

## 🎯 Problem 3: Word Search II (LC #212)

Given a board of characters and a list of words, find all words that can be formed by adjacent cells. This is the classic trie + backtracking combo.

**Why trie here?** Without a trie, you'd DFS for each word separately. With a trie, you search for ALL words simultaneously in a single DFS.

In [ ]:
def findWords(board, words):
    """
    Build trie from words, then DFS on board.
    Time: O(m*n * 4^L) where L = max word length
    """
    # Build trie
    root = TrieNode()
    for word in words:
        node = root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.word = word  # Store complete word at end node
    
    result = []
    rows, cols = len(board), len(board[0])
    
    def dfs(r, c, node):
        if r < 0 or r >= rows or c < 0 or c >= cols:
            return
        ch = board[r][c]
        if ch not in node.children:
            return
        
        child = node.children[ch]
        
        # Found a complete word?
        if hasattr(child, 'word'):
            result.append(child.word)
            del child.word  # Avoid duplicates
        
        # Mark visited
        board[r][c] = '#'
        
        # Explore 4 directions
        for dr, dc in [(0,1), (0,-1), (1,0), (-1,0)]:
            dfs(r + dr, c + dc, child)
        
        board[r][c] = ch  # Restore
        
        # Optimization: remove leaf nodes (pruning)
        if not child.children:
            del node.children[ch]
    
    for r in range(rows):
        for c in range(cols):
            dfs(r, c, root)
    
    return result

board = [
    ['o','a','a','n'],
    ['e','t','a','e'],
    ['i','h','k','r'],
    ['i','f','l','v']
]
words = ["oath", "pea", "eat", "rain"]
print(f"Found words: {findWords(board, words)}")  # ["oath", "eat"]

**The pruning optimization** (removing empty children) is important — without it, the trie keeps dead branches that slow down future searches.

---

## 🗺️ Trie Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  TRIE PATTERNS                                                      ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  STRUCTURE: Each node = {children: {char: node}, is_end: bool}      ║
║                                                                      ║
║  1. BASIC TRIE: insert, search, startsWith                          ║
║     Navigate char by char. is_end marks complete words.             ║
║                                                                      ║
║  2. WILDCARD SEARCH: '.' = try all children (DFS/backtrack)         ║
║                                                                      ║
║  3. TRIE + BOARD SEARCH: Build trie from words, DFS on grid        ║
║     Search all words simultaneously. Prune dead branches.           ║
║                                                                      ║
║  WHEN TO USE TRIE: prefix matching, autocomplete, word lists +     ║
║  searches, replacing many string comparisons with single traversal  ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 208 | Implement Trie | Medium | Core trie: insert/search/prefix |
| 211 | Add and Search Words | Medium | Wildcard with DFS backtrack |
| 212 | Word Search II | Hard | Trie + grid backtracking |
| 648 | Replace Words | Medium | Find shortest prefix in trie |
| 720 | Longest Word in Dictionary | Medium | BFS/DFS on trie |
| 1268 | Search Suggestions System | Medium | Trie + DFS for top 3 |

---

## ✅ Chapter 04 Final Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Trees = recursion. Base case: if not node: return              │
├──────────────────────────────────────────────────────────────────────┤
│  📌 BST inorder = sorted. Validate with range. Delete = successor │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Diameter / Max Path: return one side, track global both sides  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 LCA: if both sides return → current = LCA                     │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Trie: for prefix problems, wildcard, and multi-word search    │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Chapter: Graphs!

BFS, DFS, connected components, topological sort, shortest paths.

---

### 🎉 Chapter 04 Complete — Trees Mastered!
From basics to BST to tries — you've got the full tree toolkit.